# Onshape Pack and Go
Export all parts (STEP) and their linked drawings (PDF) from an Onshape assembly.

**On Google Colab:**
1. Click the 🔑 **Secrets** tab in the left sidebar
2. Add `ONSHAPE_ACCESS_KEY` and `ONSHAPE_SECRET_KEY` (from [Onshape Developer Portal](https://dev-portal.onshape.com/))
3. Paste your assembly URL in the Config cell below
4. Runtime → Run all

**Running locally:**
1. Fill in your keys in the `.env` file next to this notebook
2. Paste your assembly URL in the Config cell below
3. Run all cells

In [1]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
!pip install requests python-dotenv --quiet

In [2]:
# ── 2. Configuration — fill in your assembly URL here ───────────────────────
ASSEMBLY_URL = "https://cad.onshape.com/documents/c601fe038fc7edee8059465d/w/80217bf5820f36db2ce2c0d2/e/46b7791b4095dc77298c94b1"

In [3]:
# ── 3. Load API keys ─────────────────────────────────────────────────────────
import os

try:
    from google.colab import userdata
    ACCESS_KEY = userdata.get('ONSHAPE_ACCESS_KEY')
    SECRET_KEY = userdata.get('ONSHAPE_SECRET_KEY')
    print("✓ API keys loaded from Colab Secrets")
except Exception:
    # Local: load from .env file if present
    from dotenv import load_dotenv
    load_dotenv()
    ACCESS_KEY = os.environ.get('ONSHAPE_ACCESS_KEY', '')
    SECRET_KEY = os.environ.get('ONSHAPE_SECRET_KEY', '')
    if ACCESS_KEY:
        print("✓ API keys loaded from .env file")
    else:
        print("Running locally — keys not found in .env")

if not ACCESS_KEY or not SECRET_KEY:
    raise ValueError("API keys not found. Add ONSHAPE_ACCESS_KEY and ONSHAPE_SECRET_KEY to Colab Secrets or a .env file.")

✓ API keys loaded from .env file


In [4]:
# ── 4. Onshape API client with Basic auth ───────────────────────────────────
import requests
import time
import re

BASE_URL = 'https://cad.onshape.com'
AUTH = (ACCESS_KEY, SECRET_KEY)


def _onshape_request(method, path, query=None, body=None):
    """Make an authenticated Onshape API request using Basic auth."""
    headers = {'Accept': 'application/json'}
    if body is not None:
        headers['Content-Type'] = 'application/json'

    url = BASE_URL + '/api/v6' + path
    resp = requests.request(method, url, auth=AUTH, headers=headers,
                            params=query, json=body)
    resp.raise_for_status()
    return resp


def api_get(path, query=None):
    return _onshape_request('GET', path, query=query).json()


def api_post(path, body=None, query=None):
    return _onshape_request('POST', path, query=query, body=body).json()


def api_get_binary(path, query=None):
    return _onshape_request('GET', path, query=query).content


print('✓ Onshape client ready')


In [ ]:
# ── DEBUG: Inspect views and try export via translations ─────────────────────
import requests, json as _json

_did   = "c601fe038fc7edee8059465d"
_wvmid = "80217bf5820f36db2ce2c0d2"
BASE   = "https://cad.onshape.com/api/v6"
AUTH   = (ACCESS_KEY, SECRET_KEY)
HDR    = {"Accept": "application/json"}

drawing_eids = [
    ("PandG Hexagon Drawing 1", "ff8449d6fcd2aedf1900347c"),
    ("PandG Circle Drawing 1",  "b5e964787338acfc87fae690"),
    ("PandG Rectange Drawing 1","7ce7a9ffdf012040dd9b6055"),
]

# Print full views response for first drawing
name, eid = drawing_eids[0]
resp = requests.get(f"{BASE}/drawings/d/{_did}/w/{_wvmid}/e/{eid}/views", auth=AUTH, headers=HDR)
print(f"Full views for {name}:")
print(_json.dumps(resp.json(), indent=2)[:3000])

# Try export via translations endpoint
print("\n=== Try translations export ===")
resp2 = requests.post(
    f"{BASE}/translations",
    auth=AUTH,
    headers={**HDR, "Content-Type": "application/json"},
    json={
        "documentId": _did,
        "workspaceId": _wvmid,
        "elementId": eid,
        "formatName": "PDF",
        "storeInDocument": False,
    }
)
print(f"[{resp2.status_code}]", resp2.text[:400])


In [5]:
# ── 5. Parse assembly URL ────────────────────────────────────────────────────
import re

def parse_onshape_url(url):
    """Extract document ID, workspace/version/microversion type+ID, and element ID."""
    pattern = r"documents/([a-f0-9]+)/(w|v|m)/([a-f0-9]+)/e/([a-f0-9]+)"
    m = re.search(pattern, url)
    if not m:
        raise ValueError(f"Could not parse Onshape URL: {url}")
    did, wvm, wvmid, eid = m.group(1), m.group(2), m.group(3), m.group(4)
    return did, wvm, wvmid, eid


did, wvm, wvmid, eid = parse_onshape_url(ASSEMBLY_URL)
print(f"Document:  {did}")
print(f"Workspace: {wvmid} ({wvm})")
print(f"Element:   {eid}")

Document:  c601fe038fc7edee8059465d
Workspace: 80217bf5820f36db2ce2c0d2 (w)
Element:   46b7791b4095dc77298c94b1


In [ ]:
# ── 6. Get all unique RELEASED parts from the assembly ───────────────────────

def get_assembly_parts(did, wvm, wvmid, eid):
    """
    Walk the assembly BOM to collect all unique parts in the "Released" state.
    Returns a list of dicts with keys: partId, elementId, documentId, documentMicroversion, name.
    """
    data = api_get(
        f"/assemblies/d/{did}/{wvm}/{wvmid}/e/{eid}/bom",
        query={"bomType": "flattened", "indented": "false", "multiLevel": "false"},
    )

    # Build header lookup: propertyName -> columnId
    headers = {h["propertyName"]: h["id"] for h in data.get("headers", [])}
    name_col  = headers.get("name")
    state_col = headers.get("state")

    if state_col is None:
        print("⚠ No 'State' column found in BOM — release filtering skipped.")

    seen = set()
    parts = []
    skipped = []

    for row in data.get("rows", []):
        vals = row.get("headerIdToValue", {})

        name  = vals.get(name_col, "") or "unnamed"
        state = vals.get(state_col, "") or ""

        # Filter to Released only
        if state_col is not None and state.lower() != "released":
            skipped.append(f"{name} ({state or 'no state'})")
            continue

        src = row.get("itemSource", {})
        if not src:
            continue

        part_id   = src.get("partId")
        element_id = src.get("elementId")
        doc_id    = src.get("documentId") or did
        doc_microversion = src.get("sourceElementMicroversionId", "")

        key = (doc_id, element_id, part_id)
        if key in seen:
            continue
        seen.add(key)
        parts.append({
            "partId": part_id,
            "elementId": element_id,
            "documentId": doc_id,
            "documentMicroversion": doc_microversion,
            "wvmId": src.get("wvmId", wvmid),
            "wvmType": src.get("wvmType", wvm),
            "name": name,
        })

    if skipped:
        print(f"Skipped {len(skipped)} non-released part(s):")
        for s in skipped:
            print(f"  ✗ {s}")

    return parts


parts = get_assembly_parts(did, wvm, wvmid, eid)
print(f"\nFound {len(parts)} released part(s):")
for p in parts:
    print(f"  • {p['name']} (partId={p['partId']}, elementId={p['elementId']})")


In [7]:
# ── 7. Export each part as STEP ──────────────────────────────────────────────
import time

POLL_INTERVAL = 3   # seconds between status checks
POLL_TIMEOUT  = 300 # seconds before giving up on a translation


def _poll_translation(translation_id):
    """Poll a translation job until complete, then download the result."""
    deadline = time.time() + POLL_TIMEOUT
    while time.time() < deadline:
        status = api_get(f"/translations/{translation_id}")
        state = status.get("requestState", "")
        if state == "DONE":
            ext_data_ids = status.get("resultExternalDataIds", [])
            if not ext_data_ids:
                raise RuntimeError(f"Translation {translation_id} finished but returned no files.")
            doc_id = status["documentId"]
            ext_id = ext_data_ids[0]
            return api_get_binary(f"/documents/d/{doc_id}/externaldata/{ext_id}")
        elif state == "FAILED":
            raise RuntimeError(f"Translation {translation_id} failed: {status.get('failureReason')}")
        time.sleep(POLL_INTERVAL)
    raise TimeoutError(f"Translation {translation_id} did not complete within {POLL_TIMEOUT}s")


def export_part_step(part):
    """Export a single part as STEP via the partstudios translations API."""
    p_did  = part["documentId"]
    p_wvm  = part["wvmType"]
    p_wvmid = part["wvmId"]
    p_eid  = part["elementId"]
    part_id = part["partId"]

    body = {
        "formatName": "STEP",
        "partIds": part_id,
        "storeInDocument": False,
    }
    result = api_post(
        f"/partstudios/d/{p_did}/{p_wvm}/{p_wvmid}/e/{p_eid}/translations",
        body=body,
    )
    translation_id = result.get("id")
    if not translation_id:
        raise RuntimeError(f"No translation ID returned: {result}")
    return _poll_translation(translation_id)


step_files = {}  # name → bytes

for part in parts:
    print(f"Exporting STEP: {part['name']} ...", end=" ", flush=True)
    try:
        data = export_part_step(part)
        safe_name = re.sub(r'[^\w\-.]', '_', part['name'])
        step_files[f"{safe_name}.step"] = data
        print("✓")
    except Exception as e:
        print(f"✗ ({e})")

print(f"\nExported {len(step_files)} STEP file(s).")


In [ ]:
# ── 8. Find drawings linked to parts via "Where Used" ────────────────────────

def get_workspace_id(doc_id):
    """Look up the default workspace ID for a document."""
    try:
        return api_get(f"/documents/{doc_id}").get("defaultWorkspace", {}).get("id", "")
    except Exception:
        return ""

# Build a workspace map for each unique document
workspace_map = {did: wvmid}  # assembly doc already known
for p in parts:
    p_did = p["documentId"]
    if p_did not in workspace_map:
        ext_wid = get_workspace_id(p_did)
        if ext_wid:
            workspace_map[p_did] = ext_wid
        else:
            print(f"⚠ Could not find workspace for document {p_did}")


def get_where_used(part):
    """Return all elements that reference this part, using workspace."""
    p_did   = part["documentId"]
    p_wvmid = workspace_map.get(p_did, "")
    p_eid   = part["elementId"]
    part_id = part["partId"]
    if not p_wvmid:
        return []
    try:
        data = api_get(
            f"/parts/d/{p_did}/w/{p_wvmid}/e/{p_eid}/partid/{part_id}/whereused"
        )
        return data.get("uses", [])
    except Exception as e:
        print(f"  ⚠ whereused failed for {part['name']}: {e}")
        return []


def get_element_info(doc_id, wvm, wvmid, element_id):
    """Get metadata for a single element."""
    try:
        elements = api_get(f"/documents/d/{doc_id}/{wvm}/{wvmid}/elements",
                           query={"elementId": element_id})
        return elements[0] if elements else {}
    except Exception:
        return {}


linked_drawings = []  # (element_info, part_name, doc_id, wvm, wvmid)
seen_drawings = set()

for part in parts:
    print(f"Where used: {part['name']} ...", end=" ", flush=True)
    uses = get_where_used(part)
    drawing_uses = [u for u in uses if u.get("elementType") == "DRAWING"]
    print(f"{len(drawing_uses)} drawing(s)")

    for use in drawing_uses:
        u_did   = use.get("documentId", part["documentId"])
        u_wvmid = use.get("workspaceId") or workspace_map.get(u_did, "")
        u_wvm   = "w"
        u_eid   = use.get("elementId", "")

        key = (u_did, u_eid)
        if key in seen_drawings:
            continue
        seen_drawings.add(key)

        el = get_element_info(u_did, u_wvm, u_wvmid, u_eid)
        name = el.get("name") or u_eid
        print(f"  ✓ '{name}' in document {u_did}")
        linked_drawings.append((
            {"id": u_eid, "name": name},
            part["name"],
            u_did, u_wvm, u_wvmid
        ))

print(f"\n{len(linked_drawings)} total drawing(s) linked to released parts.")


In [ ]:
# ── 9. Export linked drawings as PDF ─────────────────────────────────────────

def export_drawing_pdf(doc_id, wvm, wvmid, drawing_eid):
    """Export a drawing element as PDF bytes."""
    body = {
        "format": "PDF",
        "storeInDocument": False,
    }
    result = api_post(
        f"/drawings/d/{doc_id}/{wvm}/{wvmid}/e/{drawing_eid}/export",
        body=body,
    )
    if "id" in result:
        return _poll_translation(result["id"])
    return result


pdf_files = {}  # name → bytes

for el, matched_parts, d_doc_id, d_wvm, d_wvmid in linked_drawings:
    d_eid = el["id"]
    d_name = el.get("name", d_eid)
    print(f"Exporting PDF: {d_name} ...", end=" ", flush=True)
    try:
        data = export_drawing_pdf(d_doc_id, d_wvm, d_wvmid, d_eid)
        safe_name = re.sub(r'[^\w\-.]', '_', d_name)
        pdf_files[f"{safe_name}.pdf"] = data
        print("✓")
    except Exception as e:
        print(f"✗ ({e})")

print(f"\nExported {len(pdf_files)} PDF file(s).")

In [ ]:
# ── 10. Package everything into a ZIP and download ───────────────────────────
import zipfile
import io

zip_name = "onshape_pack_and_go.zip"
zip_buffer = io.BytesIO()

with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
    for filename, data in step_files.items():
        zf.writestr(f"parts/{filename}", data)
    for filename, data in pdf_files.items():
        zf.writestr(f"drawings/{filename}", data)

zip_bytes = zip_buffer.getvalue()
print(f"ZIP created: {len(zip_bytes) / 1024:.1f} KB")
print(f"  parts/    — {len(step_files)} STEP file(s)")
print(f"  drawings/ — {len(pdf_files)} PDF file(s)")

# Download in Colab
try:
    from google.colab import files
    files.download_bytes(zip_name, zip_bytes)
    print(f"\n✓ Download started: {zip_name}")
except ImportError:
    # Running locally — save to disk
    with open(zip_name, "wb") as f:
        f.write(zip_bytes)
    print(f"\n✓ Saved locally: {zip_name}")